# Linear Regression Tutorial: Titanic Dataset

This notebook walks through a complete linear regression workflow using the Titanic dataset.

**Goal:** Predict passenger survival (0 or 1) using linear regression as a baseline model, and explore the relationship between features and the target variable.

> **Note:** The survival target is binary, so logistic regression is theoretically more appropriate. However, linear regression is applied here as an educational baseline — a common starting point before introducing classification-specific algorithms.

**Sections:**
1. Load & Inspect Data
2. Exploratory Data Analysis (EDA)
3. Feature Engineering
4. Preprocessing
5. Train Linear Regression Model
6. Evaluate the Model
7. Predictions on Test Set

## 1. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score, confusion_matrix, ConfusionMatrixDisplay

import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 5)

print('Libraries loaded successfully.')

## 2. Load & Inspect Data

In [ ]:
DATA_DIR = 'data/titanic'

train_df = pd.read_csv(f'{DATA_DIR}/training.csv')
test_df  = pd.read_csv(f'{DATA_DIR}/test.csv')

print(f'Training set: {train_df.shape[0]} rows, {train_df.shape[1]} columns')
print(f'Test set    : {test_df.shape[0]} rows, {test_df.shape[1]} columns')
train_df.head()

In [ ]:
train_df.info()

In [ ]:
train_df.describe()

In [ ]:
# Missing values summary
missing = train_df.isnull().sum()
missing_pct = (missing / len(train_df) * 100).round(2)
pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct}).query('`Missing Count` > 0')

## 3. Exploratory Data Analysis (EDA)

In [ ]:
# Survival rate overall
survived_rate = train_df['Survived'].value_counts(normalize=True) * 100
print(f"Survived: {survived_rate[1]:.1f}%   Did not survive: {survived_rate[0]:.1f}%")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

train_df['Survived'].value_counts().plot(kind='bar', ax=axes[0], color=['#e74c3c', '#2ecc71'], edgecolor='black')
axes[0].set_title('Survival Count')
axes[0].set_xticklabels(['Did Not Survive', 'Survived'], rotation=0)
axes[0].set_ylabel('Count')

sns.histplot(data=train_df, x='Age', hue='Survived', bins=30, kde=True, ax=axes[1])
axes[1].set_title('Age Distribution by Survival')

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

sns.barplot(data=train_df, x='Pclass', y='Survived', ax=axes[0], palette='Blues_d')
axes[0].set_title('Survival Rate by Passenger Class')
axes[0].set_ylabel('Survival Rate')

sns.barplot(data=train_df, x='Sex', y='Survived', ax=axes[1], palette='Set2')
axes[1].set_title('Survival Rate by Sex')

sns.barplot(data=train_df, x='Embarked', y='Survived', ax=axes[2], palette='Set1')
axes[2].set_title('Survival Rate by Embarkation Port')

plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap (numeric columns only)
numeric_cols = train_df.select_dtypes(include=np.number).columns.tolist()
corr = train_df[numeric_cols].corr()

plt.figure(figsize=(9, 6))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1)
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

## 4. Feature Engineering

In [ ]:
def engineer_features(df):
    df = df.copy()

    # Family size
    df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
    df['IsAlone'] = (df['FamilySize'] == 1).astype(int)

    # Title extracted from Name
    df['Title'] = df['Name'].str.extract(r',\s*([^\.]+)\.')
    rare_titles = df['Title'].value_counts()[df['Title'].value_counts() < 10].index
    df['Title'] = df['Title'].replace(rare_titles, 'Rare')
    df['Title'] = df['Title'].replace({'Mlle': 'Miss', 'Ms': 'Miss', 'Mme': 'Mrs'})

    # Age: fill missing with median per Title group
    df['Age'] = df.groupby('Title')['Age'].transform(lambda x: x.fillna(x.median()))
    df['Age'].fillna(df['Age'].median(), inplace=True)

    # Fare: fill missing with median per Pclass
    df['Fare'] = df.groupby('Pclass')['Fare'].transform(lambda x: x.fillna(x.median()))

    # Age bins
    df['AgeBin'] = pd.cut(df['Age'], bins=[0, 12, 18, 35, 60, 100],
                          labels=['Child', 'Teen', 'YoungAdult', 'Adult', 'Senior'])

    # Fare bins
    df['FareBin'] = pd.qcut(df['Fare'], q=4, labels=['Low', 'Mid', 'High', 'VeryHigh'])

    # Embarked: fill missing with mode
    df['Embarked'].fillna(df['Embarked'].mode()[0], inplace=True)

    return df

train_df = engineer_features(train_df)
test_df  = engineer_features(test_df)

print('Feature engineering complete.')
train_df[['FamilySize', 'IsAlone', 'Title', 'AgeBin', 'FareBin']].head()

## 5. Preprocessing

In [ ]:
CATEGORICAL_FEATURES = ['Sex', 'Embarked', 'Title', 'AgeBin', 'FareBin']
NUMERIC_FEATURES     = ['Pclass', 'Age', 'Fare', 'FamilySize', 'IsAlone', 'SibSp', 'Parch']
TARGET = 'Survived'

def preprocess(df, categorical_features, numeric_features, fit_scaler=None):
    df = df.copy()

    # One-hot encode categoricals
    df_encoded = pd.get_dummies(df[categorical_features + numeric_features],
                                columns=categorical_features, drop_first=True)

    # Scale numerics
    scaler = fit_scaler if fit_scaler else StandardScaler()
    df_encoded[numeric_features] = scaler.fit_transform(df_encoded[numeric_features]) \
        if not fit_scaler else scaler.transform(df_encoded[numeric_features])

    return df_encoded, scaler

X_all, scaler = preprocess(train_df, CATEGORICAL_FEATURES, NUMERIC_FEATURES)
y_all = train_df[TARGET]

X_test_final, _ = preprocess(test_df, CATEGORICAL_FEATURES, NUMERIC_FEATURES, fit_scaler=scaler)

# Align columns between train and test
X_test_final = X_test_final.reindex(columns=X_all.columns, fill_value=0)

print(f'Feature matrix shape: {X_all.shape}')
X_all.head()

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X_all, y_all, test_size=0.2, random_state=42, stratify=y_all
)

print(f'Train size: {X_train.shape[0]}  |  Validation size: {X_val.shape[0]}')

## 6. Train Linear Regression Model

In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)

print('Model trained.')
print(f'Intercept : {model.intercept_:.4f}')

In [ ]:
# Feature coefficients
coef_df = pd.DataFrame({
    'Feature': X_train.columns,
    'Coefficient': model.coef_
}).sort_values('Coefficient', key=abs, ascending=False)

plt.figure(figsize=(10, 6))
colors = ['#2ecc71' if c > 0 else '#e74c3c' for c in coef_df['Coefficient']]
plt.barh(coef_df['Feature'], coef_df['Coefficient'], color=colors, edgecolor='black')
plt.axvline(0, color='black', linewidth=0.8)
plt.title('Linear Regression Feature Coefficients')
plt.xlabel('Coefficient Value')
plt.tight_layout()
plt.show()

## 7. Evaluate the Model

In [ ]:
y_pred_raw  = model.predict(X_val)
y_pred      = (y_pred_raw >= 0.5).astype(int)  # threshold at 0.5

mse  = mean_squared_error(y_val, y_pred_raw)
rmse = np.sqrt(mse)
r2   = r2_score(y_val, y_pred_raw)
acc  = accuracy_score(y_val, y_pred)

print(f'MSE  : {mse:.4f}')
print(f'RMSE : {rmse:.4f}')
print(f'R²   : {r2:.4f}')
print(f'Accuracy (threshold 0.5): {acc:.4f}')

In [ ]:
# Cross-validation R² score
cv_scores = cross_val_score(model, X_all, y_all, cv=5, scoring='r2')
print(f'5-Fold CV R² scores : {cv_scores.round(4)}')
print(f'Mean CV R²          : {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')

In [ ]:
# Actual vs Predicted plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(y_val, y_pred_raw, alpha=0.5, color='steelblue', edgecolors='white', linewidths=0.4)
axes[0].axhline(0.5, color='red', linestyle='--', label='Decision boundary (0.5)')
axes[0].set_xlabel('Actual Survived')
axes[0].set_ylabel('Predicted Score')
axes[0].set_title('Actual vs Predicted Survival Score')
axes[0].legend()

residuals = y_val - y_pred_raw
axes[1].scatter(y_pred_raw, residuals, alpha=0.5, color='darkorange', edgecolors='white', linewidths=0.4)
axes[1].axhline(0, color='black', linestyle='--')
axes[1].set_xlabel('Predicted Score')
axes[1].set_ylabel('Residual')
axes[1].set_title('Residual Plot')

plt.tight_layout()
plt.show()

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_val, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Did Not Survive', 'Survived'])

fig, ax = plt.subplots(figsize=(6, 5))
disp.plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title('Confusion Matrix (Validation Set)')
plt.tight_layout()
plt.show()

In [ ]:
# Predicted score distribution
plt.figure(figsize=(9, 4))
sns.histplot(y_pred_raw, bins=30, kde=True, color='steelblue')
plt.axvline(0.5, color='red', linestyle='--', label='Decision boundary (0.5)')
plt.title('Distribution of Predicted Survival Scores')
plt.xlabel('Predicted Score')
plt.legend()
plt.tight_layout()
plt.show()

## 8. Predictions on Test Set

In [ ]:
test_scores = model.predict(X_test_final)
test_predictions = (test_scores >= 0.5).astype(int)

submission = pd.DataFrame({
    'PassengerId': test_df['PassengerId'],
    'Survived': test_predictions
})

submission.to_csv(f'{DATA_DIR}/submission.csv', index=False)
print(f'Submission saved. Predicted survivors: {test_predictions.sum()} / {len(test_predictions)}')
submission.head(10)

## Summary

| Metric | Value |
|--------|-------|
| Training samples | 713 |
| Validation samples | 178 |
| Model | Linear Regression |
| RMSE | — (see above) |
| R² | — (see above) |
| Accuracy (threshold 0.5) | — (see above) |

**Key takeaways:**
- `Sex_male`, `Pclass`, and `Title` are the strongest predictors of survival.
- Linear regression treats survival as a continuous score. Predictions outside [0, 1] are possible — logistic regression would bound them naturally.
- The residual plot shows non-random patterns because the true relationship is non-linear (binary outcome), which is a fundamental limitation of applying linear regression to classification tasks.

**Next steps:** Try `LogisticRegression`, `RandomForestClassifier`, or gradient boosting (XGBoost/LightGBM) to improve classification performance.